In [ ]:
import matplotlib.pyplot as plt

from rocketpy import Environment, Flight, Function, Rocket, SolidMotor, LinearGenericSurface
from rocketpy import StochasticEnvironment, StochasticRocket, StochasticFlight, MonteCarlo

plt.style.use("seaborn-v0_8-colorblind")

## Environment 

In [ ]:
# env = Environment(
#     date=(2023, 10, 12, 11),
#     latitude=39.3897,
#     longitude=-8.288964,
#     elevation=158,
#     datum="WGS84",
#     timezone="Portugal",
# )
env = Environment(
    date=(2025, 7, 26, 0),
    latitude=22.1749259,
    longitude=120.8922531,
    elevation=20,
    datum="WGS84",
    timezone="UTC",
)
env.set_atmospheric_model(
    type="Ensemble",
    file="../../data/weather/ae7f067fbd351ec2055c0e748c3e7f29.nc",
    dictionary="ECMWF",
)
env.max_expected_height = 4000
env.info()

stochastic_env = StochasticEnvironment(
    environment=env,
)
stochastic_env.visualize_attributes()

## Rocket, Motor and Aerodynamic surfaces

In [ ]:
balloon_radius = 1.5 # m

SM = SolidMotor(
    thrust_source=50,
    burn_time=0.2,
    grain_number=1,
    grain_density=100,
    grain_initial_inner_radius=0.01,
    grain_outer_radius=0.035,
    grain_initial_height=0.1,
    nozzle_radius=0.0335,
    nozzle_position=0,
    throat_radius=0.0114,
    grain_separation=0.00,
    grains_center_of_mass_position=0.2,
    dry_inertia=(0, 0, 0),
    center_of_dry_mass_position=0,
    dry_mass=0,
)
# SM.info(filename=None)

aero_model = LinearGenericSurface( reference_area = balloon_radius*balloon_radius*3.14,
                            reference_length = 1,
                            coefficients={
                                        "cL_0": Function(lambda x1, x2, x3, x4, x5, x6, x7: 0.00, ["alpha","beta","mach","reynolds","pitch_rate","yaw_rate","roll_rate"],["cL_0"]),
                                        "cQ_0": Function(lambda x1, x2, x3, x4, x5, x6, x7: 0.00, ["alpha","beta","mach","reynolds","pitch_rate","yaw_rate","roll_rate"],["cQ_0"]),
                                        "cD_0": Function(lambda x1, x2, x3, x4, x5, x6, x7: 1, ["alpha","beta","mach","reynolds","pitch_rate","yaw_rate","roll_rate"],["cD_0"]),
                                        "cm_p": Function(lambda x1, x2, x3, x4, x5, x6, x7: 0.01, ["alpha","beta","mach","reynolds","pitch_rate","yaw_rate","roll_rate"],["cm_p"]),
                                        "cm_q": Function(lambda x1, x2, x3, x4, x5, x6, x7: 0.01, ["alpha","beta","mach","reynolds","pitch_rate","yaw_rate","roll_rate"],["cm_q"]),
                                        "cm_r": Function(lambda x1, x2, x3, x4, x5, x6, x7: 0.01, ["alpha","beta","mach","reynolds","pitch_rate","yaw_rate","roll_rate"],["cm_r"]),
                                        "cn_p": Function(lambda x1, x2, x3, x4, x5, x6, x7: 0.01, ["alpha","beta","mach","reynolds","pitch_rate","yaw_rate","roll_rate"],["cn_p"]),
                                        "cn_q": Function(lambda x1, x2, x3, x4, x5, x6, x7: 0.01, ["alpha","beta","mach","reynolds","pitch_rate","yaw_rate","roll_rate"],["cn_q"]),
                                        "cn_r": Function(lambda x1, x2, x3, x4, x5, x6, x7: 0.01, ["alpha","beta","mach","reynolds","pitch_rate","yaw_rate","roll_rate"],["cn_r"]),
                                        "cl_p": Function(lambda x1, x2, x3, x4, x5, x6, x7: 0.01, ["alpha","beta","mach","reynolds","pitch_rate","yaw_rate","roll_rate"],["cl_p"]),
                                        "cl_q": Function(lambda x1, x2, x3, x4, x5, x6, x7: 0.01, ["alpha","beta","mach","reynolds","pitch_rate","yaw_rate","roll_rate"],["cl_q"]),
                                        "cl_r": Function(lambda x1, x2, x3, x4, x5, x6, x7: 0.01, ["alpha","beta","mach","reynolds","pitch_rate","yaw_rate","roll_rate"],["cl_r"]),
                            },
                            center_of_pressure=(0,0,0),
                            name="Aero Model")


Balloon = Rocket(
    volume=balloon_radius*balloon_radius*balloon_radius*3.14*4/3,
    radius=0.05,
    mass=1.2,
    inertia=(1, 1, 1),
    center_of_mass_without_motor=0.2,
    power_off_drag=0,
    power_on_drag=0,
    coordinate_system_orientation="tail_to_nose",
)

Balloon.add_motor(SM, position=0)
Balloon.add_surfaces(aero_model, positions=(0,0,0.2))
# Balloon.set_rail_buttons(0.4, 0.1)
Balloon.draw()
Balloon.prints.rocket_aerodynamics_quantities()


stochastic_rocket = StochasticRocket(
    rocket=Balloon,
    mass=0.2,
    volume=0.5,
    inertia_11=0.1,
    inertia_22=0.1,
    inertia_33=0.1,
    center_of_mass_without_motor=0,
)
stochastic_rocket.visualize_attributes()


## Flight Simulation

In [ ]:
flight = Flight(
    rocket=Balloon,
    environment=env,
    inclination=90,
    heading=180,
    rail_length=0.1,
    max_time=100,
    max_time_step=0.01,
    verbose=True,
)

flight.plots.aerodynamic_forces()
# flight.prints.apogee_conditions()
flight.plots.trajectory_3d()
# flight.plots.attitude_data()
flight.plots.linear_kinematics_data()
flight.plots.angular_kinematics_data()

In [ ]:
stochastic_flight = StochasticFlight(
    flight=flight,
    inclination=5,
    heading=90
)
stochastic_flight.visualize_attributes()

test_dispersion = MonteCarlo(
    filename="./monte_carlo_class_example",
    environment=stochastic_env,
    rocket=stochastic_rocket,
    flight=stochastic_flight,
    export_list=['t_final']
)

test_dispersion.simulate(
    number_of_simulations=10,
    append=False,
    include_function_data=False,
    parallel=False,
    n_workers=4,
)

test_dispersion.prints.all()